# **MODULE 2: Learn to Race** 🏁

---

**Welcome back.** In Part 1 we trained a car that could complete laps, but it was squiggly, occasionally hugged walls, and had no incentive to actually finish a lap cleanly.

In Part 2 we fix all of that. **Same PPO algorithm. Same network. Same simulator. Different reward function.** (The same from-scratch PPO you built in Part 1, with the same hyperparameters.)

This is the core lesson of reward shaping: the agent is already doing its job perfectly, it optimizes exactly what you reward. If the behavior is wrong, the reward function is not ideal for your objective.

---

<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/mod2n.png" width="750">


## 📋 Before You Start

### What You Will Learn
- Why reward function design matters more than algorithm choice
- How to add steering smoothness, wall safety, and centerline tracking signals
- How to measure and compare policy quality beyond just episode reward
- How to read training metrics to verify each reward term is working

### Prerequisites
- ✅ Complete Part 1 first — this notebook builds directly on those concepts
- ✅ T4 GPU runtime selected (**Runtime → Change runtime type → T4 GPU**)
- ✅ WandB account at [wandb.ai](https://wandb.ai)

### Runtime
- **Training time: ~40–45 minutes** on a Colab T4 GPU (1M steps)

---

> **📌 Note:** We use the same `F1TenthRLWrapper` class from Part 1. The only thing changing is the reward function inside `step()`.

---

## 📝 Student TODOs — Complete These Before Running Training

This notebook has **9 TODOs** to complete. Some carry over directly from Part 1 —
they are marked accordingly. If you have not done Part 1, complete those first
before attempting Part 2.

| # | TODO | Location | Note |
|---|---|---|---|
| **TODO 1** | Define the observation space | Section 7 — Wrapper `__init__` | Same as Part 1 |
| **TODO 2** | Normalize the lidar scan in `_process_obs` | Section 7 — Wrapper `_process_obs` | Same as Part 1 |
| **TODO 3** | Scale network actions to physical units | Section 7 — Wrapper `_scale_action` | Same as Part 1 |
| **TODO 4** | Choose reward shaping coefficients | Section 7 — Config block | **New in Part 2** |
| **TODO 5** | Implement the 3 new reward terms | Section 7 — Wrapper `step()` | **New in Part 2** |
| **TODO 6** | Initialize `prev_physical_steer` in `reset()` | Section 7 — Wrapper `reset()` | **New in Part 2** |
| **TODO 7** | Actor-critic forward pass | Section 7.2 — `ActorCritic.forward` | Same as Part 1 (TODO 5) |
| **TODO 8** | Compute GAE advantages | Section 7.2 — `compute_gae` | Same as Part 1 (TODO 6) |
| **TODO 9** | Choose PPO hyperparameters + launch training | Section 8 — Training | Same as Part 1 |

> **📌 TODOs 1, 2, 3, 7, 8, 9** are the same as Part 1. If you completed Part 1,
> copy your implementations here. If you are seeing Part 2 first,
> refer to the Section 1 concepts and Section 7 wrapper explanation before filling them in.

> **📌 TODOs 4, 5, 6** are new to Part 2. Read Sections 3–5 carefully before
> attempting them — each section explains one reward component you will implement.

---


---

# **Section 1 — Why Part 1 Wasn't Good Enough**

## 1.1 What We Saw After Part 1

After training for 1,000,000 steps in Part 1, our eval results looked something like this (rewards would depend on your reward scales):

```
Ep 1: reward=316.4 | steps=10000 | speed=5.98 m/s | progress=0.751 | laps=1
Ep 2: reward=316.4 | steps=10000 | speed=5.98 m/s | progress=0.751 | laps=1
Ep 3: reward=316.4 | steps=10000 | speed=5.98 m/s | progress=0.751 | laps=1
```

At first glance this looks great. But notice:
- **Every episode is identical** — reward=316.4 ± 0.0, same steps, same progress
- The policy found one fixed path and repeats it deterministically
- We have no idea if the steering was smooth, how close to walls it got, or whether it could generalize

## 1.2 The Root Cause

Our Part 1 reward only asked for two things:

```
reward = delta_progress × 10     # move forward
       + 0.03 × (speed / vmax)   # go faster
       = -30  if collision        # don't crash
```

The agent found the optimal strategy for *this reward* — not for *good racing*. Specifically:

| Problem | Why It Happened |
|---|---|
| **Squiggly steering** | No penalty for changing steering direction rapidly |
| **Hugging walls** | No cost for being close to walls (as long as no collision) |

## 1.3 What We Want

A policy that:
- ✅ Steers **smoothly** — no rapid oscillations
- ✅ Maintains a **safe margin** from walls
- ✅ Still drives **fast** and **makes forward progress**

We will add three new reward terms to achieve this. Let's go through each one.

---

# **Section 2 — Setup**

## 2.1 Install and Import

In [ ]:
# ── Install all dependencies ──────────────────────────────────────────────────
!pip install -q git+https://github.com/f1tenth/f1tenth_gym.git@dev-humble
!pip install -q stable-baselines3[extra]
!pip install -q wandb

print("Installation complete!")

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import gymnasium as gym
import numpy as np
import os
import copy
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
from collections import deque
import stable_baselines3
import f1tenth_gym

# Vectorized environment utilities only — run several simulators in parallel and
# normalize rewards. The PPO algorithm itself is our own from-scratch version.
from stable_baselines3.common.vec_env import (
    SubprocVecEnv, DummyVecEnv, VecMonitor, VecNormalize
)

print(f"PyTorch:          {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
print(f"Stable Baselines: {stable_baselines3.__version__}  (vec-env utilities only)")
print()
print("All good — let's shape some rewards!")

In [ ]:
# ── Pre-cache the track map ───────────────────────────────────────────────────
from f1tenth_gym.envs.track import Track
Track.from_track_name('Spielberg')
print("Spielberg track cached — ready to train!")

---

# **Section 3 — Reward Component 1: Steering Smoothness**

## 3.1 The Problem

In Part 1 the car steered by rapidly oscillating left and right. From the agent's perspective, this was perfectly rational — it moved forward and didn't crash, so it got rewarded. The wobbly path is actually slightly faster in some corners than a smooth arc, so the agent learned it.

On a real car this is catastrophic:
- Rapid steering changes stress the servo motor
- The sim-to-real transfer breaks down because the real car can't respond at 100Hz
- The car oscillates and eventually crashes

## 3.2 The Fix — Steering Jerk Penalty

<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/steerdelta.png" width="250">

We penalize the **change** in the car's physical steering angle (radians, from `std_state[2]`) between consecutive timesteps:

```
steer_delta = |physical_steer_now - physical_steer_prev|
penalty     = -0.05 × steer_delta
```

### Why this scale?

| Scenario | `steer_delta` | Penalty per step |
|---|---|---|
| Driving perfectly straight | 0.0 | 0.0 |
| Gentle curve (normal) | ~0.02 rad | -0.001 |
| Moderate correction | ~0.08 rad | -0.004 |
| Full lock-to-lock (−0.42 → +0.42 rad) | ~0.84 rad | -0.042 |

The penalty is **small enough** not to discourage necessary steering (e.g. turning a corner), but large enough to punish rapid oscillations that serve no purpose.

## 3.3 What to Watch

In WandB, the new metric **`racing/avg_steer_jerk`** will show this directly.

| Metric | Part 1 Baseline | Part 2 Target |
|---|---|---|
| `racing/avg_steer_jerk` | ~0.15–0.25 | < 0.05 |

---

# **Section 4 — Reward Component 2: Wall Proximity**

## 4.1 The Problem

<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/wallproximity.png" width="250">

In Part 1, the car had no cost for being close to a wall, only for actually hitting one. So it learned to drive as close to the inside wall as possible on corners (the shortest path) without caring about safety margin.

This works in simulation where collision is a binary event. On a real car:
- Sensor noise means the car might think it's 0.3m from a wall when it's actually 0.15m
- Latency means the steering command arrives 3 steps late
- The real car is slightly wider than the sim model

Any of these will turn a "close but safe" sim trajectory into a crash on the real car.

## 4.2 The Fix — Wall Proximity Penalty

We add a soft penalty that activates when the car gets within 0.5m of any wall:

```
min_scan = minimum lidar reading (closest wall distance)

if min_scan < 0.5m:
    penalty = -0.2 × (0.5 - min_scan)
```

### How the penalty scales:

| Distance to wall | Penalty per step |
|---|---|
| > 0.5m | 0.0 (no penalty) |
| 0.4m | -0.02 |
| 0.3m | -0.04 |
| 0.1m | -0.08 |
| 0.0m (touching) | -0.10 |

This creates a **soft repulsion** from walls that grows as the car gets closer, without hard-coding a minimum distance.

## 4.3 What to Watch

| Metric | Part 1 Baseline | Part 2 Target |
|---|---|---|
| `racing/avg_min_scan` | ~0.60m | > 0.80m |

---

# **Section 5 — Reward Component 3: Frenet Lateral Error**

## 5.1 The Problem — Car Picks Any Path That Avoids Walls

In Parts 1 and 2 so far, the car only needs to move forward without crashing. It has no concept of *where* on the track it should be, just that it should avoid the walls.

The result is that the car picks whatever path minimizes crashes while maximizing progress. This often means cutting corners aggressively, drifting wide on exits, or taking completely inconsistent lines through the same corner on different laps.

On a real car this matters because:
- Inconsistent lines make the policy unpredictable and harder to tune
- Wide exits increase the risk of hitting the outer wall
- The centerline is typically the safest path when no raceline is available

---

## 5.2 The Frenet Frame

The **Frenet frame** converts the car's `(x, y)` global position into track-relative coordinates:

```
Global frame:      x, y     ← absolute position in the world
Frenet frame:      s, ey    ← position relative to the track

    s  = distance traveled along the track (same as our progress metric)
    ey = lateral error — how far LEFT or RIGHT of the centerline
         ey = 0.0  → perfectly on centerline
         ey = +0.3 → 0.3m to the left of centerline
         ey = -0.5 → 0.5m to the right of centerline
```

The F1TENTH gym computes this for us and returns it in `raw_obs["agent_0"]["frenet_pose"]`.

---

## 5.3 What Does "Centerline" Mean Here?

The `ey` value is measured relative to whatever the gym's `Track` object loaded as its reference line. This is the same source as our waypoints:

| Map | Reference Line Used |
|---|---|
| Spielberg | `Spielberg_raceline.csv` — the actual racing line |
| Your custom map | Whatever CSV you placed in the gym's maps folder |

So **if you provide a raceline CSV, the penalty rewards following the raceline.** If you provide a centerline CSV, it rewards staying in the middle of the corridor. The behavior the penalty produces depends entirely on the reference line you give it.

---

## 5.4 The Fix — Lateral Error Penalty

We add a soft penalty that activates only when the car drifts more than 0.3m from the reference line:

```python
frenet = raw_obs["agent_0"].get("frenet_pose", None)
if frenet is not None:
    ey = float(frenet[1])        # lateral error in metres
    if abs(ey) > 0.3:
        reward -= 0.3 * (abs(ey) - 0.3)
```

### Why a 0.3m Tolerance?

Requiring `ey = 0` exactly would be too strict — the car needs freedom to navigate corners. The 0.3m dead zone allows normal racing maneuvers while still penalizing wide, inconsistent lines.

### How the Penalty Scales:

| Lateral Error | Penalty per Step |
|---|---|
| < 0.3m (tolerance zone) | 0.0 (no penalty) |
| 0.4m | -0.03 |
| 0.5m | -0.06 |
| 0.8m | -0.15 |

---

## 5.5 What to Watch

After adding this penalty, `racing/avg_lateral_err` should decrease over training — the car learning to stay closer to the reference line.

| Metric | Without Frenet | With Frenet Target |
|---|---|---|
| `racing/avg_lateral_err` | ~0.4–0.6m | < 0.25m |
| Lap consistency | variable path | more consistent line |

> ⚠️ **Note:** If `frenet_pose` is not available in your gym version, the penalty silently does nothing (`if frenet is not None` guards against this). Check your gym version if you don't see the lateral error improving.


---

# **Section 6 — Full Shaped Reward**

## 6.1 All Components Together

Here is the complete Part 2 reward function:

```
reward = delta_progress × 10.0          # move forward (main signal)
       + 0.03 × (speed / vmax)          # go faster (anti-crawl)
       - 0.05 × |physical_steer_now - physical_steer_prev|  # steer smoothly (NEW)
       - 0.2 × max(0, 0.5 - min_scan)  # stay away from walls (NEW)
       - 0.3 × max(0, |ey| - 0.3)       # stay near centerline (NEW)
       = -30.0 if collision              # don't crash (same as Part 1)
```

## 6.2 Reward Scale Justification

| Term | Typical Value | Purpose | Relative Weight |
|---|---|---|---|
| Progress | +0.01 to +0.05/step | Primary goal | **dominant** |
| Speed bonus | +0.005 to +0.03/step | Secondary | small |
| Steer penalty | -0.001 to -0.04/step | Shape behavior | small |
| Wall penalty | 0 to -0.10/step | Safety | small |
| Lateral penalty | 0 to -0.15/step | Line consistency | small |
| Crash | -30.0 once | Hard constraint | large but rare |

The shaping terms (steer, wall, lateral) are deliberately kept small — they guide behavior without overriding the core progress signal. If they were too large, the car would stop moving to avoid the penalties.

## 6.3 The Wrapper — What Changed from Part 1

Only **4 things** changed in the code:

1. `self.prev_physical_steer` added to `__init__` and set from `std_state[2]` in `reset()`
2. Steering penalty computed in `step()` from the physical steering angle change `|std_state[2] - self.prev_physical_steer|`
3. Wall penalty computed from `min_scan`
4. Lateral error penalty computed from the Frenet `ey`

Everything else — observation, action scaling, PPO hyperparameters, network architecture — is identical to Part 1, including the same from-scratch PPO (network, GAE, clipped loss).

---

# **Section 7 — Implementation**

## 7.1 The Part 2 Environment Wrapper

In [ ]:
# ── F1TenthRLWrapper v2 — with reward shaping ─────────────────────────────────

# ═══════════════════════════════════════════════════════════════════════════
# ✏️  TODO 4 — Reward Shaping Coefficient Configuration
# ═══════════════════════════════════════════════════════════════════════════

# Before implementing the reward terms, decide on the scale for each one.
# These constants will be used inside step() below.
#
# Think carefully about the RELATIVE sizes:
#   - The progress reward gives roughly 0.01-0.05 per step
#   - Your shaping penalties should guide behavior without dominating progress
#   - If a penalty is too large the car will stop moving to avoid it
#   - If a penalty is too small it will have no effect
#
# STEER_SMOOTH_COEF   — multiplied by |Δsteering_angle| each step
#                       Too high → car avoids steering entirely, drives straight
#                       Typical range: 0.01 to 0.1
#
# WALL_PROX_COEF      — multiplied by (threshold - min_scan) when near a wall
#                       Too high → car drives slowly away from walls
#                       Typical range: 0.05 to 0.2
#
# WALL_PROX_THRESHOLD — distance (metres) at which wall penalty activates
#                       Below this → penalty grows; above this → no penalty
#                       Typical range: 0.3m to 0.8m
#
# LATERAL_ERR_COEF    — multiplied by (|ey| - tolerance) when off centerline
#                       Too high → car hugs centerline but slows in corners
#                       Typical range: 0.2 to 1.0
#
# LATERAL_TOLERANCE   — lateral error (metres) allowed before penalty activates
#                       Allows natural cornering without penalty
#                       Typical range: 0.2m to 0.5m

# ───────────────────────────────────────────────────────────────────────────





# ═══════════════════════════════════════════════════════════════════════════


class F1TenthRLWrapper(gym.Env):
    """
    Part 2 wrapper — same structure as Part 1, new reward function.

    Observation (110-dim, all in [0,1]):
        [0:108]  lidar scan — 108 beams, gym clips at 10m
        [108]    speed, normalized
        [109]    steering angle, normalized

    Action (2-dim, in [-1,1]):
        [0]  steering  → [-max_steer, +max_steer] rad
        [1]  speed     → [min_speed, max_speed] m/s

    Reward (Part 2 — shaped):
        +delta_progress * 10.0              forward progress (same as Part 1)
        +0.03 * (speed/vmax)                speed bonus (same as Part 1)
        -STEER_SMOOTH_COEF * |Δsteer|       steering smoothness  [NEW - TODO 5a]
        -WALL_PROX_COEF * wall_error        wall proximity       [NEW - TODO 5b]
        -LATERAL_ERR_COEF * lateral_error   centerline tracking  [NEW - TODO 5c]
        -30.0 on collision                  crash penalty (same as Part 1)
    """

    def __init__(self, map_name="Spielberg", num_beams=108, render_mode=None):
        super().__init__()

        from f1tenth_gym.envs.env_config import (
            EnvConfig, SimulationConfig, ObservationConfig, ResetConfig,
        )
        from f1tenth_gym.envs.lidar.config import LiDARConfig
        from f1tenth_gym.envs.observation import ObservationType
        from f1tenth_gym.envs.reset import ResetStrategy
        from f1tenth_gym.envs.integrators import IntegratorType
        from f1tenth_gym.envs.dynamic_models import DynamicModel

        env_config = EnvConfig(
            map_name=map_name,
            num_agents=1,
            simulation_config=SimulationConfig(
                timestep=0.01,
                integrator=IntegratorType.RK4,
                dynamics_model=DynamicModel.ST,
                max_laps=5,
            ),
            observation_config=ObservationConfig(type=ObservationType.DIRECT),
            lidar_config=LiDARConfig(
                num_beams=num_beams,
                range_max=10.0,
                noise_std=0.0,
            ),
            reset_config=ResetConfig(strategy=ResetStrategy.RL_RANDOM_STATIC),
            render_enabled=(render_mode is not None),
        )

        self.env = gym.make(
            "f1tenth_gym:f1tenth-v0",
            config=env_config,
            render_mode=render_mode,
        )

        self.num_beams    = num_beams
        self.lidar_clip   = 10.0
        self.max_speed    = 6.0
        self.min_speed    = 1.5
        self.max_steer    = float(self.env.action_space.high[0][0])
        self.max_steps    = 10000
        self.current_step = 0

        try:
            self.track_length = self.env.unwrapped.track.centerline.length
        except Exception:
            self.track_length = None

        self.waypoints           = self._extract_waypoints()
        self.prev_progress       = 0.0
        self.start_progress      = 0.0
        self.prev_lap_count      = 0
        self.prev_physical_steer = 0.0

        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 1 — Define the Observation Space
        # ═══════════════════════════════════════════════════════════════

        # Same as Part 1 — if you completed Part 1, copy your solution here.
        # If you are starting from Part 2, refer to Section 7 explanation:
        #   110 values total, all normalized to [0, 1]

        # ───────────────────────────────────────────────────────────────





        # ═══════════════════════════════════════════════════════════════

        self.action_space = gym.spaces.Box(
            low=-1.0, high=1.0, shape=(2,), dtype=np.float32
        )

    # ----------------------------------------------------------

    def _extract_waypoints(self):
        """Fallback only — used if frenet_pose unavailable."""
        try:
            track = self.env.unwrapped.track
            line  = getattr(track, "raceline", None) or getattr(track, "centerline", None)
            if line is not None:
                wps = np.column_stack([line.xs, line.ys])
                print(f"[ENV] Loaded {len(wps)} waypoints (fallback)")
                return wps
        except Exception as e:
            print(f"[ENV] WARNING: {e}")
        t = np.linspace(0, 2 * np.pi, 100, endpoint=False)
        return np.column_stack([5 * np.cos(t), 5 * np.sin(t)])

    def _get_progress(self, raw_obs):
        """
        Returns progress 0.0→1.0 along the track.
        Primary: gym frenet s / track_length. Fallback: waypoint search.
        """
        try:
            frenet = raw_obs["agent_0"].get("frenet_pose", None)
            if frenet is not None and self.track_length:
                s = float(frenet[0])
                return (s % self.track_length) / self.track_length
        except Exception:
            pass
        state   = raw_obs["agent_0"]["std_state"]
        x, y    = float(state[0]), float(state[1])
        dists   = np.linalg.norm(self.waypoints - np.array([x, y]), axis=1)
        return np.argmin(dists) / len(self.waypoints)

    def _process_obs(self, raw_obs):
        agent = raw_obs["agent_0"]
        scan  = agent["scan"]   # already (108,) and clipped to 10m

        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 2 — Normalize the Lidar Scan
        # ═══════════════════════════════════════════════════════════════

        # scan values are in metres (0 to 10). Divide by self.lidar_clip
        # so 10m → 1.0, then cast to float32.

        # ───────────────────────────────────────────────────────────────





        # ═══════════════════════════════════════════════════════════════


        state      = agent["std_state"]
        speed_norm = np.clip(float(state[3]) / self.max_speed, 0.0, 1.0)
        steer_norm = np.clip(
            (float(state[2]) + self.max_steer) / (2 * self.max_steer), 0.0, 1.0
        )
        return np.concatenate([
            scan.astype(np.float32),
            np.array([speed_norm, steer_norm], dtype=np.float32)
        ])

    def _scale_action(self, action):

        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 3 — Scale Network Actions to Physical Commands
        # ═══════════════════════════════════════════════════════════════

        # action[0], action[1] are in [-1, 1]. Map them to physical units:
        #   steer: [-1,1] → [-max_steer, +max_steer]  (multiply by max_steer)
        #   speed: [-1,1] → [min_speed, max_speed]

        # ───────────────────────────────────────────────────────────────





        # ═══════════════════════════════════════════════════════════════

        return np.array([[steer, speed]], dtype=np.float32)

    # ----------------------------------------------------------

    def reset(self, seed=None, options=None):
        raw_obs, info     = self.env.reset(seed=seed)
        self.current_step = 0
        self.prev_lap_count = 0

        state = raw_obs["agent_0"]["std_state"]

        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 6 — Initialize prev_physical_steer
        # ═══════════════════════════════════════════════════════════════

        # The steering smoothness penalty measures how much the steering
        # ANGLE changed between steps. We need to track the previous step's
        # physical steering angle so we can compute the delta each step.
        #
        # At reset, initialize it from the car's actual starting angle —
        # not zero, because the car may start mid-corner at a random position.
        #
        # Hint: std_state[2] is the actual steering angle in radians

        # ───────────────────────────────────────────────────────────────





        # ═══════════════════════════════════════════════════════════════

        self.start_progress = self._get_progress(raw_obs)
        self.prev_progress  = self.start_progress

        return self._process_obs(raw_obs), info

    def step(self, action):
        physical_action = self._scale_action(action)
        raw_obs, _, done, truncated, info = self.env.step(physical_action)
        self.current_step += 1

        state     = raw_obs["agent_0"]["std_state"]
        speed     = float(state[3])
        collision = bool(raw_obs["agent_0"].get("collision", False))
        scan_raw  = raw_obs["agent_0"]["scan"]
        min_scan  = float(np.min(scan_raw))

        lap_count     = int(raw_obs["agent_0"].get("lap_count", 0))
        lap_completed = lap_count > self.prev_lap_count
        self.prev_lap_count = lap_count

        # Progress (frenet-based, same as Part 1)
        progress       = self._get_progress(raw_obs)
        delta_progress = progress - self.prev_progress
        if delta_progress < -0.5:
            delta_progress += 1.0

        # Base reward from Part 1 (progress + speed bonus)
        reward  = delta_progress * 10.0
        reward += 0.03 * (speed / self.max_speed)

        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 5 — Implement the 3 New Reward Shaping Terms
        # ═══════════════════════════════════════════════════════════════

        # You have read Sections 3-5 — now implement each component.
        # Use the constants you defined in TODO 4 above.
        # Variables available:
        #   state[2]     → current physical steering angle (rad)
        #   min_scan     → closest wall distance (metres)
        #   frenet[1]    → lateral error from centerline ey (metres)
        #   collision    → True if car hit a wall this step

        # ── ✏️  TODO 5a: Steering Smoothness Penalty ────────────────────

        # Penalize the change in PHYSICAL steering angle between steps.
        # Use std_state[2] (actual angle in radians), not action[0].
        # Larger delta → the car is steering jerkily → apply a penalty.
        # Update self.prev_physical_steer for the next step.
        # Apply STEER_SMOOTH_COEF * steer_delta

        # ───────────────────────────────────────────────────────────────






        # ── ✏️  TODO 5b: Wall Proximity Penalty ─────────────────────────

        # Apply a soft penalty when the car is too close to a wall.
        # The penalty should grow as the car gets closer.
        # Only activate when min_scan is below WALL_PROX_THRESHOLD.
        # Apply WALL_PROX_COEF * (threshold - min_scan)

        # ───────────────────────────────────────────────────────────────






        frenet      = raw_obs["agent_0"].get("frenet_pose", None)
        lateral_err = 0.0
        if frenet is not None:
            ey = float(frenet[1])
            lateral_err = abs(ey)

        # ── ✏️  TODO 5c: Frenet Lateral Error Penalty ───────────────────

        # Penalize the car for drifting away from the track centerline.
        # ey = frenet_pose[1] is the lateral error in metres (lateral_err above).
        # Only activate beyond LATERAL_TOLERANCE to allow normal cornering.
        # Apply LATERAL_ERR_COEF * (lateral_err - tolerance)

        # ───────────────────────────────────────────────────────────────






        # Crash penalty — overrides everything, ends episode (same as Part 1)
        if collision:
            reward = -30.0
            done   = True

        # ═══════════════════════════════════════════════════════════════

        if self.current_step >= self.max_steps:
            truncated = True

        distance_traveled = progress - self.start_progress
        if distance_traveled < 0:
            distance_traveled += 1.0

        info["speed"]         = speed
        info["collision"]     = collision
        info["progress"]      = distance_traveled
        info["min_scan"]      = min_scan
        info["lap_completed"] = lap_completed
        info["steer_delta"]   = steer_delta if "steer_delta" in dir() else 0.0
        info["lateral_err"]   = lateral_err

        self.prev_progress = progress
        return self._process_obs(raw_obs), float(reward), done, truncated, info

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()


print("F1TenthRLWrapper v2 defined!")
print("Remember: complete all TODOs before running training!")


## 7.2 PPO from Scratch — Same as Part 1

We reuse the exact from-scratch PPO you built in Part 1: the same actor-critic
network, the same GAE, and the same clipped surrogate loss (inside the training
loop in Section 8). Nothing about the algorithm changes in Part 2 — only the reward.

- **Actor-Critic network (TODO 7)** — two independent [64, 64] Tanh MLPs. The actor outputs
  the Gaussian mean $\mu(s)$ (with a learned, state-independent $\log\sigma$), the critic outputs $V(s)$.
- **GAE (TODO 8)** — computed backwards through each rollout:

$$\delta_t = r_t + \gamma\, V(s_{t+1})\,(1 - d_t) - V(s_t) \qquad A_t = \delta_t + \gamma\lambda\,(1 - d_t)\,A_{t+1} \qquad R_t = A_t + V(s_t)$$

The new **`racing/avg_steer_jerk`** metric (average steering change per step — lower is
smoother) is logged directly inside the training loop, alongside the Part 1 metrics.

> **📌 Note:** TODOs 7 and 8 are the same as TODO 5 and TODO 6 in Part 1 — paste in your own
> working versions if you have them.


In [ ]:
# ── Actor-Critic Network ──────────────────────────────────────────────────────
# Two INDEPENDENT [64,64] MLPs — no shared trunk. Sharing weights between the
# actor and critic would let policy gradients and value gradients interfere
# with the same parameters, hurting both objectives. Orthogonal
# initialization with a small gain on the action head keeps initial actions
# near zero, which is important for early training stability.

def _orthogonal_init(layer, gain):
    nn.init.orthogonal_(layer.weight, gain=gain)
    nn.init.constant_(layer.bias, 0.0)

class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        # Actor — fully independent MLP, own weights
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.actor_mean = nn.Linear(hidden, act_dim)

        # Critic — fully independent MLP, own weights (no sharing with the actor)
        self.critic_body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.critic = nn.Linear(hidden, 1)

        # Learnable log standard deviation (state-independent)
        self.log_std = nn.Parameter(torch.zeros(act_dim))

        # Orthogonal init gains: sqrt(2) for hidden layers, 0.01 for the
        # action output (near-zero initial actions), 1.0 for the value output
        for layer in self.actor:
            if isinstance(layer, nn.Linear): _orthogonal_init(layer, gain=2 ** 0.5)
        for layer in self.critic_body:
            if isinstance(layer, nn.Linear): _orthogonal_init(layer, gain=2 ** 0.5)
        _orthogonal_init(self.actor_mean, gain=0.01)
        _orthogonal_init(self.critic, gain=1.0)

    def forward(self, obs):
        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 7 — Actor-Critic Forward Pass
        # ═══════════════════════════════════════════════════════════════

        # Pass obs through the actor to get the action mean and through the
        # critic to get the state value, then build a Normal distribution
        # from the mean and the learned log-std.
        # Return: dist, value

        # ───────────────────────────────────────────────────────────────





        # ═══════════════════════════════════════════════════════════════
        pass  # remove once you've implemented the steps above

    @torch.no_grad()
    def act(self, obs, deterministic=False):
        """Sample an action (or take the mean if deterministic)."""
        dist, value = self.forward(obs)
        action = dist.mean if deterministic else dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        return action, log_prob, value


print('ActorCritic network defined (separate actor/critic MLPs, orthogonal init).')

In [ ]:
# ── GAE — Generalized Advantage Estimation ────────────────────────────────────
def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    """
    rewards, values, dones : tensors of shape (T, N)  [T steps, N envs]
    last_value             : tensor (N,) — V(s_T) for bootstrapping
    Returns advantages and returns, both (T, N).
    """
    T, N = rewards.shape
    advantages = torch.zeros_like(rewards)
    last_adv = torch.zeros(N, device=rewards.device)

    # ═══════════════════════════════════════════════════════════════
    # ✏️  TODO 8 — Compute GAE Advantages
    # ═══════════════════════════════════════════════════════════════

    # Loop backwards from t = T-1 to 0. At each step, take the next value
    # (last_value at the end), discount it only if the episode didn't end,
    # compute the TD residual, then add the decayed advantage from step t+1
    # and store the result in advantages[t].

    # ───────────────────────────────────────────────────────────────





    # ═══════════════════════════════════════════════════════════════

    returns = advantages + values
    return advantages, returns


print('compute_gae defined.')

## 7.3 Environment Builders and Training Config

In [ ]:
# ── Environment builders — identical to Part 1 ────────────────────────────────

def make_env(map_name, rank, seed=0):
    def _init():
        env = F1TenthRLWrapper(map_name=map_name)
        env.reset(seed=seed + rank)
        return env
    return _init


def make_training_env(map_name, num_envs):
    if num_envs == 1:
        env = DummyVecEnv([make_env(map_name, 0)])
    else:
        env = SubprocVecEnv([make_env(map_name, i) for i in range(num_envs)])
    env = VecMonitor(env)
    env = VecNormalize(
        env,
        norm_obs    = False,
        norm_reward = True,
        clip_reward = 10.0,
        gamma       = 0.99,
    )
    return env


def make_eval_env(map_name):
    env = DummyVecEnv([make_env(map_name, 99)])
    env = VecMonitor(env)
    env = VecNormalize(
        env,
        norm_obs    = False,
        norm_reward = True,
        clip_reward = 10.0,
        gamma       = 0.99,
        training    = False,
    )
    return env


print("Environment builders defined.")

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
# 🔧  IMPORTANT - Change these to experiment

MAP_NAME    = "Spielberg"
NUM_ENVS    = 4
TOTAL_STEPS = 1_000_000      # same as Part 1 for fair comparison
DEVICE      = "cuda"

print(f"Training config:")
print(f"  Map:    {MAP_NAME}")
print(f"  Envs:   {NUM_ENVS}")
print(f"  Steps:  {TOTAL_STEPS:,}")
print(f"  Device: {DEVICE}")

---

# **Section 7.4 — WandB Login**

WandB tracks all metrics in real time. Run the cell below and paste your API key
from https://wandb.ai/authorize when prompted. This is needed for both the sweep and training.


In [ ]:
# ── WandB login ───────────────────────────────────────────────────────────────

# Paste your API key directly to avoid the interactive prompt
# Get it from: wandb.ai/authorize
# WANDB_API_KEY = "your_key_here"
# wandb.login(key=WANDB_API_KEY)

wandb.login()

---

# **Section 7.5 — Reward Shaping Sweep** *(Run Before Full Training)*

## Find Your Best Shaping Coefficients with WandB Sweeps

Part 2 adds three shaping coefficients (steering smoothness, wall proximity, lateral error).
Rather than guessing all of them, use this sweep to compare combinations with short 50k-step runs.
WandB tries every combination and logs results to a parallel coordinates plot.


<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/hyperparam_sweep.png" width="650">


**How to use:**
1. ✅ Complete **TODO 4 + TODO 5** in the wrapper first — the sweep uses your reward implementation
2. Edit the `parameters` lists below to the values you want to compare
3. Run the cell — WandB launches one run per combination
4. Open the sweep link that prints — results update live
5. Use the best values in your wrapper, then run full training in Section 8

> **⚠️ Requires the reward TODOs complete.** The sweep trains using your wrapper's `step()`.


In [ ]:
# ── WandB Reward Shaping Sweep ────────────────────────────────────────────────
# Edit the value lists below — every combination will be trained and compared.
# Each run is a short 50k-step version of the same from-scratch PPO loop used in Section 8.

# ── Define values to sweep (the 3 shaping coefficients) ──────────────────────
sweep_config = {
    "method": "grid",
    "metric": {"name": "eval/mean_reward", "goal": "maximize"},
    "parameters": {
        "steer_smooth_coef": {"values": [0.02, 0.05]},
        "wall_prox_coef":    {"values": [0.1, 0.2]},
        "lateral_err_coef":  {"values": [0.3, 0.5]},
    }
}

SWEEP_STEPS          = 50_000
SWEEP_ENVS           = 2
SWEEP_N_STEPS        = 1024      # rollout length per env per update
SWEEP_EPOCHS         = 5
SWEEP_MINIBATCH      = 128
SWEEP_EVAL_EPISODES  = 3
SWEEP_EVAL_MAX_STEPS = 5000

# ── Training function — uses YOUR wrapper reward ─────────────────────────────
def sweep_train():
    global STEER_SMOOTH_COEF, WALL_PROX_COEF, LATERAL_ERR_COEF

    # reinit=True: start a fresh WandB run for every sweep combination.
    run = wandb.init(reinit=True)
    cfg = wandb.config
    print(f"  steer={cfg.steer_smooth_coef} | wall={cfg.wall_prox_coef} | lateral={cfg.lateral_err_coef}")

    # The wrapper reads these module-level constants inside step(), so write
    # this run's values to them before the envs are built.
    _orig = (STEER_SMOOTH_COEF, WALL_PROX_COEF, LATERAL_ERR_COEF)
    STEER_SMOOTH_COEF = cfg.steer_smooth_coef
    WALL_PROX_COEF    = cfg.wall_prox_coef
    LATERAL_ERR_COEF  = cfg.lateral_err_coef

    s_train = s_eval = None
    try:
        torch.manual_seed(42); np.random.seed(42)
        device = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')

        # DummyVecEnv (same process) so the constants set above are the ones the envs read
        s_train = DummyVecEnv([make_env(MAP_NAME, i) for i in range(SWEEP_ENVS)])
        s_train = VecMonitor(s_train)
        s_train = VecNormalize(s_train, norm_obs=False, norm_reward=True,
                               clip_reward=10.0, gamma=0.99)

        s_eval  = DummyVecEnv([make_env(MAP_NAME, 99)])
        s_eval  = VecMonitor(s_eval)
        s_eval  = VecNormalize(s_eval, norm_obs=False, norm_reward=False,
                               clip_reward=10.0, gamma=0.99, training=False)

        obs_dim = s_train.observation_space.shape[0]
        act_dim = s_train.action_space.shape[0]
        net = ActorCritic(obs_dim, act_dim).to(device)
        opt = optim.Adam(net.parameters(), lr=3e-4, eps=1e-5)

        obs = torch.tensor(s_train.reset(), dtype=torch.float32, device=device)
        n_updates = SWEEP_STEPS // (SWEEP_N_STEPS * SWEEP_ENVS)

        for _update in range(n_updates):
            b_obs_l, b_act_l, b_logp_l, b_val_l, b_rew_l, b_done_l = [], [], [], [], [], []
            for _t in range(SWEEP_N_STEPS):
                action, logp, value = net.act(obs)
                next_obs, reward, dones, _ = s_train.step(np.clip(action.cpu().numpy(), -1.0, 1.0))
                b_obs_l.append(obs); b_act_l.append(action)
                b_logp_l.append(logp); b_val_l.append(value)
                b_rew_l.append(torch.tensor(reward, dtype=torch.float32, device=device))
                b_done_l.append(torch.tensor(dones, dtype=torch.float32, device=device))
                obs = torch.tensor(next_obs, dtype=torch.float32, device=device)

            with torch.no_grad():
                _, last_value = net(obs)
            values = torch.stack(b_val_l)
            advantages, returns = compute_gae(torch.stack(b_rew_l), values, torch.stack(b_done_l),
                                              last_value, gamma=0.99, lam=0.95)

            b_obs  = torch.stack(b_obs_l).reshape(-1, obs_dim)
            b_act  = torch.stack(b_act_l).reshape(-1, act_dim)
            b_logp = torch.stack(b_logp_l).reshape(-1)
            b_adv  = advantages.reshape(-1)
            b_ret  = returns.reshape(-1)

            idxs = np.arange(b_obs.shape[0])
            for _epoch in range(SWEEP_EPOCHS):
                np.random.shuffle(idxs)
                for start in range(0, len(idxs), SWEEP_MINIBATCH):
                    mb = idxs[start:start + SWEEP_MINIBATCH]
                    dist, value = net(b_obs[mb])
                    new_logp = dist.log_prob(b_act[mb]).sum(-1)
                    entropy  = dist.entropy().sum(-1).mean()
                    mb_adv   = b_adv[mb]
                    mb_adv   = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)
                    ratio    = torch.exp(new_logp - b_logp[mb])
                    policy_loss = -torch.min(ratio * mb_adv,
                                             torch.clamp(ratio, 0.8, 1.2) * mb_adv).mean()
                    value_loss  = ((value - b_ret[mb]) ** 2).mean()
                    loss = policy_loss + 0.5 * value_loss - 0.001 * entropy
                    opt.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(net.parameters(), 0.5)
                    opt.step()

        # ── Deterministic evaluation (raw, un-normalized reward) ─────────────
        ep_rewards, ep_progress, speeds, collisions = [], [], [], []
        for _ep in range(SWEEP_EVAL_EPISODES):
            eobs = s_eval.reset()
            ep_r, ep_p = 0.0, 0.0
            for _t in range(SWEEP_EVAL_MAX_STEPS):
                eaction, _, _ = net.act(torch.tensor(eobs, dtype=torch.float32, device=device),
                                        deterministic=True)
                eobs, er, edones, einfos = s_eval.step(np.clip(eaction.cpu().numpy(), -1.0, 1.0))
                ep_r += float(er[0])
                speeds.append(einfos[0].get("speed", 0.0))
                ep_p = float(einfos[0].get("progress", ep_p))   # distance traveled (laps)
                if edones[0]:
                    collisions.append(float(einfos[0].get("collision", False)))
                    break
            else:
                collisions.append(0.0)
            ep_rewards.append(ep_r)
            ep_progress.append(ep_p)

        wandb.log({
            "eval/mean_reward":    float(np.mean(ep_rewards)),
            "eval/mean_progress":  float(np.mean(ep_progress)),
            "eval/avg_speed":      float(np.mean(speeds)),
            "eval/collision_rate": float(np.mean(collisions)),
        })
    finally:
        if s_train is not None: s_train.close()
        if s_eval  is not None: s_eval.close()
        STEER_SMOOTH_COEF, WALL_PROX_COEF, LATERAL_ERR_COEF = _orig   # restore your chosen values
        run.finish()

# ── Launch ───────────────────────────────────────────────────────────────────
n_combos = (len(sweep_config['parameters']['steer_smooth_coef']['values']) *
            len(sweep_config['parameters']['wall_prox_coef']['values']) *
            len(sweep_config['parameters']['lateral_err_coef']['values']))

sweep_id = wandb.sweep(sweep_config, project="f1tenth-rl-shaping-sweep")
print(f"Sweep: {n_combos} combinations × {SWEEP_STEPS:,} steps each")
print(f"View at: https://wandb.ai/{wandb.Api().default_entity}/f1tenth-rl-shaping-sweep/sweeps/{sweep_id}")
wandb.agent(sweep_id, function=sweep_train, count=None)
print()
print("✅ Sweep complete — open the WandB link above to compare results.")
print("   Use the Parallel Coordinates plot to find the best combination.")
print("   Update those values in your wrapper (TODO 4), re-run that cell, then run full training in Section 8.")

---

# **Section 8 — Training**

## 8.1 Training Dashboard

When you run the TRAIN cell, you get **both** monitoring options:

| WandB Dashboard | Inline Plots |
|---|---|
| A link prints at the top — open it in a new browser tab for the full interactive dashboard | 6 charts render below the cell and refresh live as training runs |

First run the cell below to define the live-plot helper, then run the TRAIN cell.


In [ ]:
# ── Live Inline Training Plots ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import clear_output

_live = {"steps": [], "rew": [], "collision": [], "speed": [],
         "laps": [], "loss": [], "std": []}

def draw_live_plots(total_steps):
    if len(_live['steps']) < 2:
        return
    BG = '#0d1117'
    fig = plt.figure(figsize=(14, 7), facecolor=BG)
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.6, wspace=0.35)
    plots = [
        (gs[0,0], _live['rew'],       'ep_rew_mean ↑',         '#00bcd4'),
        (gs[0,1], _live['collision'], 'collision_rate ↓',      '#f44336'),
        (gs[1,0], _live['speed'],     'avg_speed m/s ↑',       '#69ff47'),
        (gs[1,1], _live['laps'],      'lap_completion_rate ↑', '#ff9800'),
        (gs[2,0], _live['loss'],      'train/loss',            '#e91e63'),
        (gs[2,1], _live['std'],       'train/std ↓',           '#9c27b0'),
    ]
    for spec, data, title, color in plots:
        ax = fig.add_subplot(spec)
        ax.set_facecolor('#1a1a2e')
        valid = [(s, v) for s, v in zip(_live['steps'], data) if v is not None]
        if valid:
            xs, ys = zip(*valid)
            ax.plot(xs, ys, color=color, lw=2, marker='o', markersize=3)
            ax.fill_between(xs, ys, alpha=0.15, color=color)
            lo, hi = min(ys), max(ys)
            pad = (hi - lo) * 0.1 or 0.1
            ax.set_ylim(lo - pad, hi + pad)
        ax.set_title(title, color='white', fontsize=9, pad=4)
        ax.set_xlabel('Steps', color='#888', fontsize=7)
        ax.tick_params(colors='#888', labelsize=7)
        for spine in ax.spines.values(): spine.set_edgecolor('#333')

    step = _live['steps'][-1]
    pct  = step / total_steps * 100
    wandb_url = wandb.run.url if wandb.run else 'not connected'
    fig.suptitle(
        f'PPO from Scratch — Part 2  |  Step {step:,} / {total_steps:,}  ({pct:.1f}%)'
        f'   |   WandB → {wandb_url}',
        color='white', fontsize=9, x=0.02, ha='left')
    clear_output(wait=True)
    plt.show()
    plt.close(fig)

print('Live plot helper defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PPO from Scratch — Part 2 Training (Run This Single Cell)
# ══════════════════════════════════════════════════════════════════════════════
# Same from-scratch PPO as Part 1: collect rollouts → compute GAE → update with
# the clipped PPO loss. Only the reward (inside the wrapper) has changed.
#
# Metric definitions:
#   - collision_rate / lap_completion_rate = CUMULATIVE totals since training
#     started (never reset), not a moving window
#   - avg_speed / avg_min_scan / avg_steer_jerk / avg_lateral_err = averaged
#     over the ONE episode that just ended, logged the instant it completes
#   - rollout/ep_rew_mean / ep_len_mean   = rolling mean over the last 100
#     completed episodes
#   - train/* metrics                     = averaged over all minibatches in
#     the update
#   - eval/*                              = periodic deterministic evaluation
#     on a separate env
# ──────────────────────────────────────────────────────────────────────────────

# ── 1. Hyperparameters ────────────────────────────────────────────────────────

# ═══════════════════════════════════════════════════════════════════════════
# ✏️  TODO 9 — Choose PPO Hyperparameters
# ═══════════════════════════════════════════════════════════════════════════

# Same as Part 1 — copy your values here. Keeping them identical means any
# improvement comes purely from reward shaping, not a different algorithm config.

# ─────────────────────────────────────────────────────────────────────────────
LR            =               # TODO 9
N_STEPS       =               # TODO 9 — steps collected per env per update
N_EPOCHS      = 10            # optimization epochs per update
MINIBATCH     = 256
GAMMA         =               # TODO 9
GAE_LAMBDA    = 0.95
CLIP_EPS      =               # TODO 9 — PPO clip range ε
ENT_COEF      =               # TODO 9 — entropy bonus
VF_COEF       = 0.5           # value loss weight
MAX_GRAD      = 0.5
# ═══════════════════════════════════════════════════════════════════════════

EVAL_FREQ     = 50_000        # how often (in env steps) to run a deterministic eval pass
EVAL_EPISODES = 5

# ── Seed everything — for reproducible runs ───────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── 2. Initialize WandB (prints the live dashboard link) ──────────────────────
run = wandb.init(
    project = 'f1tenth-rl-lecture1',
    name    = f'ppo_scratch_part2_{MAP_NAME}_{TOTAL_STEPS//1000}k',
    config  = dict(version='v2_reward_shaping', map=MAP_NAME, num_envs=NUM_ENVS,
                   total_steps=TOTAL_STEPS, n_steps=N_STEPS, n_epochs=N_EPOCHS,
                   minibatch=MINIBATCH, gamma=GAMMA, gae_lambda=GAE_LAMBDA,
                   clip_eps=CLIP_EPS, ent_coef=ENT_COEF, lr=LR, net_arch='[64,64] Tanh',
                   steer_smooth_coef=STEER_SMOOTH_COEF, wall_prox_coef=WALL_PROX_COEF,
                   wall_prox_threshold=WALL_PROX_THRESHOLD,
                   lateral_err_coef=LATERAL_ERR_COEF, lateral_tolerance=LATERAL_TOLERANCE),
)
wandb.define_metric('global_step')
wandb.define_metric('*', step_metric='global_step')
print(f'🔗 WandB live dashboard → {run.url}')
print('   (open in a new tab to watch full metrics; inline plots appear below)\n')

# ── 3. Create envs (parallel rollouts + reward normalization) ─────────────────
train_env = make_training_env(MAP_NAME, NUM_ENVS)
eval_env  = make_eval_env(MAP_NAME)   # separate env for periodic deterministic eval
obs_dim   = train_env.observation_space.shape[0]
act_dim   = train_env.action_space.shape[0]

# ── 4. Build network + optimizer ──────────────────────────────────────────────
device = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')
net    = ActorCritic(obs_dim, act_dim).to(device)
opt    = optim.Adam(net.parameters(), lr=LR, eps=1e-5)   # eps=1e-5 for numerical stability
print(net)
print(f'Parameters: {sum(p.numel() for p in net.parameters()):,}\n')

# ── Metric tracking ───────────────────────────────────────────────────────────
episode_count    = 0     # cumulative — never reset
total_collisions = 0     # cumulative — never reset
total_laps       = 0     # cumulative — never reset

cur_rew      = np.zeros(NUM_ENVS)                 # per-episode accumulators
cur_spd      = [[] for _ in range(NUM_ENVS)]      # reset after each episode ends
cur_scan     = [[] for _ in range(NUM_ENVS)]
cur_jerk     = [[] for _ in range(NUM_ENVS)]      # NEW in Part 2 — steering smoothness
cur_lat      = [[] for _ in range(NUM_ENVS)]      # NEW in Part 2 — distance from centerline
cur_col      = np.zeros(NUM_ENVS, dtype=bool)
cur_lap      = np.zeros(NUM_ENVS, dtype=int)
cur_len      = np.zeros(NUM_ENVS, dtype=int)
cur_progress = np.zeros(NUM_ENVS)   # overwritten every step with the current episode's progress

ep_rew_window = deque(maxlen=100)   # rolling 100-episode window
ep_len_window = deque(maxlen=100)
last_ep       = {}                 # metrics of the most recently finished episode

_live.update({k: [] for k in _live})   # reset live plot storage
update_speeds = []   # for the local inline plot only — all speeds seen this update

obs = train_env.reset()
obs = torch.tensor(obs, dtype=torch.float32, device=device)
global_step    = 0
last_eval_step = 0
n_updates      = TOTAL_STEPS // (N_STEPS * NUM_ENVS)

print(f'Training {TOTAL_STEPS:,} steps = {n_updates} updates on {NUM_ENVS} envs ({device}) — plots update below\n')

for update in range(n_updates):
    # ── 5a. Collect rollout ───────────────────────────────────────────────────
    mb_obs, mb_act, mb_logp, mb_val, mb_rew, mb_done = [], [], [], [], [], []
    update_speeds = []

    for _ in range(N_STEPS):
        action, logp, value = net.act(obs)
        act_np = action.cpu().numpy()
        # The Gaussian sample can fall outside [-1, 1] even though that's the
        # valid action range — clip before stepping the env, but keep the
        # UNCLIPPED action for the log-prob used in the PPO update below.
        clipped_np = np.clip(act_np, -1.0, 1.0)
        next_obs, reward, dones, infos = train_env.step(clipped_np)

        # VecNormalize (outermost) returns the NORMALIZED reward, which is what
        # we train on. For episode reward TRACKING, we want the RAW reward.
        raw_reward = train_env.get_original_reward()

        mb_obs.append(obs)
        mb_act.append(action)
        mb_logp.append(logp)
        mb_val.append(value)
        mb_rew.append(torch.tensor(reward, dtype=torch.float32, device=device))
        mb_done.append(torch.tensor(dones, dtype=torch.float32, device=device))

        cur_rew += raw_reward
        cur_len += 1
        for i, inf in enumerate(infos):
            spd = inf.get('speed', 0.0)
            cur_spd[i].append(spd)
            cur_scan[i].append(inf.get('min_scan', 0.0))
            cur_jerk[i].append(inf.get('steer_delta', 0.0))
            cur_lat[i].append(inf.get('lateral_err', 0.0))
            update_speeds.append(spd)
            if 'progress' in inf: cur_progress[i] = inf['progress']
            if inf.get('collision', False): cur_col[i] = True
            if inf.get('lap_completed', False): cur_lap[i] += 1

        # ── Bootstrap truncated (not crashed) episodes before GAE ─────────────
        for i, d in enumerate(dones):
            if d and infos[i].get('TimeLimit.truncated', False):
                term_obs = torch.tensor(
                    infos[i]['terminal_observation'], dtype=torch.float32, device=device
                ).unsqueeze(0)
                with torch.no_grad():
                    _, term_value = net(term_obs)
                mb_rew[-1][i] += GAMMA * term_value.item()

        # ── Log per-episode metrics the INSTANT each episode ends ─────────────
        for i, d in enumerate(dones):
            if d:
                episode_count    += 1
                total_collisions += int(cur_col[i])
                total_laps       += cur_lap[i]
                ep_rew_window.append(cur_rew[i])
                ep_len_window.append(cur_len[i])

                last_ep = {
                    'racing/avg_speed':       float(np.mean(cur_spd[i]))  if cur_spd[i]  else 0.0,
                    'racing/avg_min_scan':    float(np.mean(cur_scan[i])) if cur_scan[i] else 0.0,
                    'racing/avg_steer_jerk':  float(np.mean(cur_jerk[i])) if cur_jerk[i] else 0.0,
                    'racing/avg_lateral_err': float(np.mean(cur_lat[i]))  if cur_lat[i]  else 0.0,
                }
                wandb.log({
                    **last_ep,
                    'racing/track_progress':      float(cur_progress[i]),
                    'racing/collision_rate':      total_collisions / episode_count,
                    'racing/lap_completion_rate': total_laps / episode_count,
                    'racing/episode_count':       episode_count,
                    'rollout/ep_rew_mean':        float(np.mean(ep_rew_window)),
                    'rollout/ep_len_mean':        float(np.mean(ep_len_window)),
                    'rollout/ep_len_std':         float(np.std(ep_len_window)) if len(ep_len_window) > 1 else 0.0,
                    'global_step':                global_step,
                })

                cur_rew[i] = 0.0; cur_spd[i] = []; cur_scan[i] = []
                cur_jerk[i] = []; cur_lat[i] = []
                cur_col[i] = False; cur_lap[i] = 0; cur_len[i] = 0

        obs = torch.tensor(next_obs, dtype=torch.float32, device=device)
        global_step += NUM_ENVS

    # ── 5b. Compute GAE ───────────────────────────────────────────────────────
    with torch.no_grad():
        _, last_value = net(obs)
    rewards = torch.stack(mb_rew)
    values  = torch.stack(mb_val)
    dones_t = torch.stack(mb_done)
    advantages, returns = compute_gae(rewards, values, dones_t, last_value,
                                      gamma=GAMMA, lam=GAE_LAMBDA)

    # Flatten (T, N, ...) → (T*N, ...)
    b_obs     = torch.stack(mb_obs).reshape(-1, obs_dim)
    b_act     = torch.stack(mb_act).reshape(-1, act_dim)
    b_logp    = torch.stack(mb_logp).reshape(-1)
    b_adv     = advantages.reshape(-1)
    b_ret     = returns.reshape(-1)
    b_val_old = values.reshape(-1)   # pre-update value predictions — for explained_variance

    # ── 5c. PPO update epochs (clipped surrogate loss, same as Part 1) ────────
    batch_size = b_obs.shape[0]
    idxs = np.arange(batch_size)
    epoch_losses, epoch_pg, epoch_vf, epoch_ent, epoch_clipfrac, epoch_kl = [], [], [], [], [], []
    for epoch in range(N_EPOCHS):
        np.random.shuffle(idxs)
        for start in range(0, batch_size, MINIBATCH):
            mb = idxs[start:start + MINIBATCH]
            dist, value = net(b_obs[mb])
            new_logp = dist.log_prob(b_act[mb]).sum(-1)
            entropy  = dist.entropy().sum(-1).mean()

            # Normalize advantages fresh for THIS minibatch
            mb_adv = b_adv[mb]
            mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

            ratio       = torch.exp(new_logp - b_logp[mb])
            surr1       = ratio * mb_adv
            surr2       = torch.clamp(ratio, 1-CLIP_EPS, 1+CLIP_EPS) * mb_adv
            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss  = ((value - b_ret[mb]) ** 2).mean()
            loss        = policy_loss + VF_COEF * value_loss - ENT_COEF * entropy

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD)
            opt.step()

            epoch_losses.append(loss.item())
            epoch_pg.append(policy_loss.item())
            epoch_vf.append(value_loss.item())
            epoch_ent.append(entropy.item())
            with torch.no_grad():
                epoch_clipfrac.append((torch.abs(ratio - 1.0) > CLIP_EPS).float().mean().item())
                log_ratio = new_logp - b_logp[mb]
                epoch_kl.append(torch.mean((torch.exp(log_ratio) - 1) - log_ratio).item())

    # ── 5d. train/* metrics — averaged over the whole update ──────────────────
    y_true = b_ret.cpu().numpy()
    y_pred = b_val_old.cpu().numpy()
    var_y  = np.var(y_true)
    explained_var = float('nan') if var_y == 0 else 1.0 - np.var(y_true - y_pred) / var_y
    current_std   = net.log_std.exp().mean().item()

    wandb.log({
        'train/loss':                 float(np.mean(epoch_losses)),
        'train/policy_gradient_loss': float(np.mean(epoch_pg)),
        'train/value_loss':           float(np.mean(epoch_vf)),
        'train/std':                  current_std,
        'train/explained_variance':   explained_var,
        'train/learning_rate':        LR,
        'train/entropy_loss':         -float(np.mean(epoch_ent)),
        'train/clip_range':           CLIP_EPS,
        'train/clip_fraction':        float(np.mean(epoch_clipfrac)),
        'train/approx_kl':            float(np.mean(epoch_kl)),
        'global_step':                global_step,
    })

    # ── 5e. Periodic evaluation ───────────────────────────────────────────────
    if global_step - last_eval_step >= EVAL_FREQ:
        last_eval_step = global_step
        # Sync eval_env's reward-normalization stats from train_env so
        # eval/mean_reward is on the same scale as training's reward.
        eval_env.ret_rms = copy.deepcopy(train_env.ret_rms)

        eval_rewards, eval_lens = [], []
        for _ in range(EVAL_EPISODES):
            eobs = eval_env.reset()
            edone, erew, elen = False, 0.0, 0
            while not edone:
                eobs_t = torch.tensor(eobs, dtype=torch.float32, device=device)
                with torch.no_grad():
                    eaction, _, _ = net.act(eobs_t, deterministic=True)
                eobs, ereward, edones, _ = eval_env.step(np.clip(eaction.cpu().numpy(), -1.0, 1.0))
                erew += float(ereward[0]); elen += 1; edone = bool(edones[0])
            eval_rewards.append(erew); eval_lens.append(elen)
        wandb.log({
            'eval/mean_reward':    float(np.mean(eval_rewards)),
            'eval/mean_ep_length': float(np.mean(eval_lens)),
            'global_step':         global_step,
        })
        print(f'  [eval @ {global_step:,}] reward={np.mean(eval_rewards):.1f} '
              f'ep_len={np.mean(eval_lens):.0f}')

    # ── 5f. Live inline plot (local only — not wandb) ─────────────────────────
    if episode_count > 0:
        _live['steps'].append(global_step)
        _live['rew'].append(float(np.mean(ep_rew_window)))
        _live['collision'].append(total_collisions / episode_count)
        _live['speed'].append(float(np.mean(update_speeds)) if update_speeds else 0.0)
        _live['laps'].append(total_laps / episode_count)
        _live['loss'].append(float(np.mean(epoch_losses)))
        _live['std'].append(current_std)
        draw_live_plots(TOTAL_STEPS)
        print(f'Update {update+1}/{n_updates} | step {global_step:,} | episodes={episode_count} | '
              f'ep_rew_mean={np.mean(ep_rew_window):.1f} | collision_rate={total_collisions/episode_count:.2f} | '
              f'lap_rate={total_laps/episode_count:.2f} | '
              f'steer_jerk={last_ep.get("racing/avg_steer_jerk", 0.0):.4f} | loss={np.mean(epoch_losses):.4f}')

# ── 6. Save + final stats ─────────────────────────────────────────────────────
os.makedirs('models_v2', exist_ok=True)
torch.save(net.state_dict(), 'models_v2/ppo_scratch_part2.pt')
train_env.save('models_v2/vecnormalize.pkl')
print()
print('Saved → models_v2/ppo_scratch_part2.pt')
print('Saved → models_v2/vecnormalize.pkl')

print()
print('─' * 60)
print('  Final Training Stats')
print('─' * 60)
for label, val in [
    ('Episode reward  ', float(np.mean(ep_rew_window)) if ep_rew_window else None),
    ('Collision rate  ', total_collisions / episode_count if episode_count else None),
    ('Avg speed (m/s) ', last_ep.get('racing/avg_speed')),
    ('Lap completion  ', total_laps / episode_count if episode_count else None),
    ('Steer jerk      ', last_ep.get('racing/avg_steer_jerk')),
    ('Train loss      ', float(np.mean(epoch_losses))),
    ('Policy std      ', current_std),
]:
    if val is not None:
        print(f'  {label}: {val:.4f}')
print('─' * 60)
print(f'  Full dashboard → {run.url}')

wandb.finish()
train_env.close()
eval_env.close()

---

# **Section 9 — Monitoring the Difference**

## 9.1 What Healthy Shaped Reward Training Looks Like

With reward shaping, you should see these patterns in WandB:

### ✅ Metrics That Should Improve vs Part 1

| Metric | What to Look For |
|---|---|
| `racing/avg_steer_jerk` | Should decrease steadily — agent learning to steer smoothly |
| `racing/avg_min_scan` | Should increase — agent keeping more distance from walls |
| `racing/lap_completion_rate` | Should rise — smoother, safer driving means more completed laps |

### ⚠️ Warning Signs Specific to Reward Shaping

| Warning Sign | What It Means | What To Do |
|---|---|---|
| `avg_speed` drops significantly | Wall penalty too strong — car staying still to avoid penalty | Reduce `WALL_PROX_COEF` from 0.2 to 0.1 |
| `track_progress` goes down | Smoothness penalty too strong — car stops to avoid jerk | Reduce `STEER_SMOOTH_COEF` from 0.05 to 0.02 |
| `ep_rew_mean` much lower than Part 1 | Penalties outweighing progress signal | Check penalty scales |
| No change vs Part 1 | Penalties too small to affect behavior | Increase penalty weights |

## 9.2 Exercises

1. **Change `STEER_SMOOTH_COEF` from 0.05 to 0.10.** Does steering become smoother? Does the car slow down to reduce jerk?
2. **Remove the wall penalty entirely.** How does `avg_min_scan` change over training?

---

# **Section 10 — Evaluate and Compare**

## 10.1 Part 2 Evaluation

The eval now reports four quality metrics:
- **`jerk`** — average steering change per step (lower = smoother)
- **`min_scan`** — average closest wall distance (higher = safer)
- **`progress`** — fraction of lap completed
- **`laps`** — full laps completed

These are the numbers that matter for real-car deployment.

In [ ]:
# ── Load and evaluate Part 2 model ────────────────────────────────────────────

model_path = "models_v2/ppo_scratch_part2.pt"
print(f"Loading Part 2 model: {model_path}")

env      = F1TenthRLWrapper(map_name=MAP_NAME)
eval_net = ActorCritic(env.observation_space.shape[0], env.action_space.shape[0]).to('cpu')
eval_net.load_state_dict(torch.load(model_path, map_location='cpu'))
eval_net.eval()

NUM_EVAL_EPISODES = 5
all_rewards     = []
all_speeds      = []
all_progress    = []
all_steer_jerks = []
all_min_scans   = []
all_laps        = []

print()
print(f"Running {NUM_EVAL_EPISODES} evaluation episodes...")
print("─" * 75)

for ep in range(NUM_EVAL_EPISODES):
    obs, _       = env.reset()
    total_reward = 0.0
    done         = False
    steps        = 0
    speeds, steer_jerks, min_scans = [], [], []
    laps = 0

    while not done:
        with torch.no_grad():
            action, _, _ = eval_net.act(torch.tensor(obs, dtype=torch.float32), deterministic=True)
        obs, reward, terminated, truncated, info = env.step(np.clip(action.numpy(), -1.0, 1.0))
        total_reward += reward
        done          = terminated or truncated
        steps        += 1
        speeds.append(info.get("speed", 0.0))
        steer_jerks.append(info.get("steer_delta", 0.0))
        min_scans.append(info.get("min_scan", 0.0))
        if info.get("lap_completed", False):
            laps += 1

    avg_spd  = np.mean(speeds)
    avg_jerk = np.mean(steer_jerks)
    avg_scan = np.mean(min_scans)
    progress = info.get("progress", 0)
    crashed  = info.get("collision", False)

    all_rewards.append(total_reward)
    all_speeds.append(avg_spd)
    all_progress.append(progress)
    all_steer_jerks.append(avg_jerk)
    all_min_scans.append(avg_scan)
    all_laps.append(laps)

    status = "CRASH" if crashed else ("TIMEOUT" if steps >= 10000 else "DONE")
    print(f"  Ep {ep+1}: {status} | reward={total_reward:7.1f} | "
          f"speed={avg_spd:.2f} m/s | laps={laps} | "
          f"jerk={avg_jerk:.4f} | scan={avg_scan:.3f}m")

print("─" * 75)
print(f"  Average reward:      {np.mean(all_rewards):.1f} ± {np.std(all_rewards):.1f}")
print(f"  Average speed:       {np.mean(all_speeds):.2f} m/s")
print(f"  Average progress:    {np.mean(all_progress):.3f}")
print(f"  Avg steer jerk:      {np.mean(all_steer_jerks):.4f}  (lower = smoother)")
print(f"  Avg min scan:        {np.mean(all_min_scans):.3f} m  (higher = safer)")
print(f"  Total laps:          {sum(all_laps)}")

env.close()

## 10.2 Record Eval Video in Colab

Colab has no display — we use `render_mode="rgb_array"` with `VecNormalize` loaded
to capture actual top-down driving frames as an MP4.

> **Note:** Requires both `models_v2/ppo_scratch_part2.pt` and `models_v2/vecnormalize.pkl`


In [ ]:
# ── Record eval video — Using F1TENTH simulator rendering ────────────────────
!pip install -q imageio imageio-ffmpeg

import imageio

print("Loading model and environment...")
vid_net = ActorCritic(obs_dim, act_dim).to('cpu')
vid_net.load_state_dict(torch.load("models_v2/ppo_scratch_part2.pt", map_location="cpu"))
vid_net.eval()

# Create environment with rgb_array rendering + VecNormalize stats
vec_env = DummyVecEnv([lambda: F1TenthRLWrapper(map_name=MAP_NAME, render_mode="rgb_array")])
vec_env = VecNormalize.load("models_v2/vecnormalize.pkl", vec_env)
vec_env.training    = False
vec_env.norm_reward = False

# Record episode
print("Recording episode...")
frames = []
obs    = vec_env.reset()
done   = False
step   = 0

while not done and step < 10000:
    with torch.no_grad():
        action, _, _     = vid_net.act(torch.tensor(obs, dtype=torch.float32), deterministic=True)
    obs, _, dones, info  = vec_env.step(np.clip(action.numpy(), -1.0, 1.0))
    done  = dones[0]
    step += 1
    if step % 5 == 0:
        frame = vec_env.envs[0].render()
        if frame is not None:
            frames.append(frame)

vec_env.close()

if frames:
    print(f"✅ Captured {len(frames)} frames from {step} steps")
    imageio.mimsave('eval_render_v2.mp4', frames, fps=20)
    print("✅ Saved → eval_render_v2.mp4")
    from google.colab import files
    files.download('eval_render_v2.mp4')
else:
    print("❌ No frames captured — check models_v2/vecnormalize.pkl exists")

In [ ]:
!zip -r models_archive.zip /content/models_v2

In [ ]:

# The 3 files needed to run locally
files.download('models_v2/ppo_scratch_part2.pt')
files.download('models_v2/vecnormalize.pkl')

## 10.3 Part 1 vs Part 2 Comparison Table

Fill in your actual numbers from both runs below.

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
# Fill in your Part 1 numbers from the eval you ran in that notebook.
# Part 2 numbers are filled in automatically from the eval above.


# ── Part 1 baseline (fill these in from your Part 1 eval) ────────────────────

p1_reward     = 316.4   # ep_rew_mean from Part 1 eval
p1_speed      = 5.98    # avg_speed
p1_steer_jerk = 0.20    # estimate — Part 1 didn't log this, typical value
p1_min_scan   = 0.60    # avg_min_scan from Part 1 training logs
p1_lap_rate   = 0.08    # lap_completion_rate from Part 1 training logs

# ── Part 2 results (computed from eval above) ─────────────────────────────────
p2_reward     = np.mean(all_rewards)
p2_speed      = np.mean(all_speeds)
p2_steer_jerk = np.mean(all_steer_jerks)
p2_min_scan   = np.mean(all_min_scans)
p2_lap_rate   = sum(all_laps) / NUM_EVAL_EPISODES

print("=" * 65)
print(f"  {'Metric':<25} {'Part 1':>12} {'Part 2':>12} {'Better?':>10}")
print("=" * 65)

def arrow(p1, p2, higher_is_better=True):
    if higher_is_better:
        return "↑" if p2 > p1 * 1.05 else ("↓" if p2 < p1 * 0.95 else "→")
    else:
        return "↓" if p2 < p1 * 0.95 else ("↑" if p2 > p1 * 1.05 else "→")

print(f"  {'ep_rew_mean':<25} {p1_reward:>12.1f} {p2_reward:>12.1f} {arrow(p1_reward, p2_reward):>10}")
print(f"  {'avg_speed (m/s)':<25} {p1_speed:>12.2f} {p2_speed:>12.2f} {arrow(p1_speed, p2_speed):>10}")
print(f"  {'avg_steer_jerk':<25} {p1_steer_jerk:>12.4f} {p2_steer_jerk:>12.4f} {arrow(p1_steer_jerk, p2_steer_jerk, higher_is_better=False):>10}")
print(f"  {'avg_min_scan (m)':<25} {p1_min_scan:>12.3f} {p2_min_scan:>12.3f} {arrow(p1_min_scan, p2_min_scan):>10}")
print(f"  {'lap_completion_rate':<25} {p1_lap_rate:>12.3f} {p2_lap_rate:>12.3f} {arrow(p1_lap_rate, p2_lap_rate):>10}")
print("=" * 65)

---

# **Section 11 — Domain Randomization: Bridging Sim to Real**

## 11.1 Why Your Trained Policy May Fail on the Real Car

Everything we trained so far ran in a perfect simulation — no sensor noise, exact physics, zero latency. The real RoboRacer is messier:

| Sim Assumption | Real-Car Reality |
|---|---|
| Lidar readings are exact | ±2–5cm noise per beam |
| Physics exactly match model | Tire friction varies with floor surface |
| Steering responds instantly | ~30–50ms servo latency |
| Speed command is exact | Motor speed drifts with battery voltage |
| Car dimensions exact | Real car slightly wider than sim model |

This gap between simulation and reality is called the **sim-to-real transfer problem**. Domain randomization is the standard solution.

---

## 11.2 What Domain Randomization Does

Instead of training in one fixed simulation, we **randomize physical parameters** each episode so the policy learns to work across a range of conditions — including the real car's conditions.

```
Each episode reset:
    friction coefficient  ← Uniform(0.8, 1.2)   # floor surface varies
    lidar noise std       ← Uniform(0.0, 0.02)  # sensor noise varies
    action delay          ← Uniform(0, 3 steps) # servo latency varies
    car mass              ← Uniform(3.0, 4.0)   # battery weight varies
```

The policy never sees the same physics twice. It learns robust behavior instead of overfitting to one exact configuration.

---

## 11.3 Adding Sensor Noise (Simple Version)

The easiest domain randomization to add — just corrupt the lidar readings in `_process_obs()`:

```python
def _process_obs(self, raw_obs):
    scan = raw_obs["agent_0"]["scan"]
    indices = np.linspace(0, len(scan)-1, self.num_beams).astype(int)
    scan = scan[indices]
    scan = np.clip(scan, 0.0, self.lidar_clip) / self.lidar_clip

    # Domain randomization: add Gaussian noise to lidar
    if self.training:   # only during training, not eval
        noise_std = np.random.uniform(0.0, 0.02)   # random noise level each episode
        scan = np.clip(scan + np.random.normal(0, noise_std, scan.shape), 0.0, 1.0)

    # ... rest of obs processing
```

---

## 11.4 Adding Action Delay

Real servos have latency. Simulating this during training forces the policy to be robust to delayed feedback:

```python
def __init__(self, ...):
    ...
    self.action_buffer = []          # stores recent actions
    self.action_delay  = np.random.randint(0, 4)  # 0-3 steps delay

def step(self, action):
    self.action_buffer.append(action)
    # Execute the action from N steps ago
    delayed_action = self.action_buffer[-1 - self.action_delay]
    physical_action = self._scale_action(delayed_action)
    raw_obs, _, done, truncated, info = self.env.step(physical_action)
    ...
```

---

## 11.5 Curriculum Domain Randomization

Rather than applying full randomization from step 1 (which makes learning harder), **curriculum DR** gradually increases the randomization as training progresses:

| Training Phase | Noise Std | Action Delay | Friction Range |
|---|---|---|---|
| 0–1M steps | 0.0 (off) | 0 steps | 1.0 (fixed) |
| 1–3M steps | 0.005 | 0–1 steps | 0.9–1.1 |
| 3–5M steps | 0.01 | 0–2 steps | 0.8–1.2 |
| 5M+ steps | 0.02 | 0–3 steps | 0.7–1.3 |

The configs in the `f1tenth_rl_humble` repo (`sim2real_e2e.yaml`, `sim2real_fast.yaml`) use exactly this curriculum approach for real-car deployment.

---

## 11.6 What to Expect

Adding domain randomization will:
- **Slow initial learning** — the policy sees harder, noisier environments early on
- **Improve final robustness** — `collision_rate` may be slightly higher in sim but much lower on real car
- **Reduce sim-to-real gap** — the policy that looks 'good enough' in sim actually works on real hardware

> **For this notebook:** We kept domain randomization out to focus on reward shaping concepts. If you are deploying on the real RoboRacer, add sensor noise and action delay before exporting your model. The configs in the framework README show the exact parameters used for Levine Hall deployment.


---

# **Section 12 — Summary & What's Next**

## 12.1 What Reward Shaping Gave Us

We made **zero changes** to:
- The PPO algorithm
- The neural network architecture
- The observation space
- The number of training steps

We only changed the **reward function** — and got a meaningfully better policy:
- Smoother steering (lower `avg_steer_jerk`)
- Safer wall margins (higher `avg_min_scan`)
- Better lap completion (higher `lap_completion_rate`)

This is the power of reward shaping: **the agent is already doing its job perfectly — it optimizes exactly what you reward.**

## 12.2 Remaining Limitations

Even with shaped rewards, there are things our policy still doesn't do:

| Limitation | Why |
|---|---|
| **Doesn't follow the optimal racing line** | We reward progress, not raceline adherence |
| **Speed varies a lot in corners** | No explicit speed profile reward |
| **Can't handle dynamic obstacles** | Single-agent training only |
| **May not transfer to real car** | No domain randomization, no sim-to-real tuning |

## 12.3 Coming Up in Lecture 2

Lecture 2 moves beyond basic PPO into advanced RL applications:

| Topic | What You'll Learn |
|---|---|
| **Raceline Optimization** | Train the agent to follow the minimum-time path, not just any path |
| **Adaptive Control** | Use RL to compensate for real-car dynamics that differ from sim |
| **Obstacle Avoidance** | Multi-agent training — your car vs an opponent |

---

> **💾 Save your models:** Download `models_v2/ppo_scratch_part2.pt` and `models_v2/vecnormalize.pkl` from the Colab file browser. You'll need both for Lecture 2.

In [ ]:
# ── Optional: Save models to Google Drive ─────────────────────────────────────

# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil, os
# os.makedirs('/content/drive/MyDrive/f1tenth_models/part2', exist_ok=True)
# shutil.copy('models_v2/ppo_scratch_part2.pt',   '/content/drive/MyDrive/f1tenth_models/part2/')
# shutil.copy('models_v2/vecnormalize.pkl',       '/content/drive/MyDrive/f1tenth_models/part2/')
# print("Models saved to Google Drive!")

#print("Uncomment the lines above to save your Part 2 model to Google Drive.")